# 01 — Data preparation (Helsinki, 2016–2021)

Input: the FMI Helsinki PV file (`FMI_Helsinki_PV.csv`) from
https://doi.org/10.57707/fmi-b2share.c7b3ec9f19324a6497639d6d8c0cde84, placed in `data/raw/`.
Output: `data/Helsinki_MultiYear.csv`.

In [1]:
import os, sys, logging, warnings
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
from pvlib.solarposition import get_solarposition
from pvlib.location import Location

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
log = logging.getLogger("helsinki")


@dataclass(frozen=True)
class HelsinkiConfig:
    raw_path: str = os.path.join("data", "raw", "FMI_Helsinki_PV.csv")
    output_path: str = os.path.join("data", "Helsinki_MultiYear.csv")

    header_row: int = 67          # metadata lines before the header
    separator: str = ";"
    years: tuple = (2016, 2017, 2018, 2019, 2020, 2021)

    site_name: str = "Helsinki"
    nominal_capacity_w: float = 20_000.0   # inverter AC rating
    latitude: float = 60.204
    longitude: float = 24.961

    source_columns: dict = field(default_factory=lambda: {
        "pv_inv_out":          "Power_Actual_W",
        "GLOBA_PT1M_AVG(:31)": "Irradiance_POA",
        "GLOB_PT1M_AVG":       "GHI",
        "TA_PT1M_AVG(:31)":    "Temp_Ambient",
        "WS_PT10M_AVG":        "Wind_Speed",
        "RH_PT1M_AVG":         "Humidity",
        "TTECH_PT1M_AVG(:32)": "Temp_Module",
        "vis_SnoP":            "Snow_Indicator",
    })
    model_features: tuple = ("Irradiance_POA", "GHI", "Temp_Ambient", "Wind_Speed", "Humidity")
    diagnostic_only: tuple = ("Temp_Module", "Solar_Elevation", "Clearness_Index", "Snow_Indicator")

    max_irradiance: float = 1400.0
    temp_bounds: tuple = (-45.0, 40.0)
    wind_bounds: tuple = (0.0, 40.0)
    dropout_min_elevation: float = 8.0     # irradiance ≈ 0 above this elevation = sensor dropout

    interpolate_limit_hours: int = 3
    climatology_window_days: int = 7


CFG = HelsinkiConfig()

## Load and resample to hourly

In [2]:
def load_raw(cfg: HelsinkiConfig) -> pd.DataFrame:
    df = pd.read_csv(cfg.raw_path, sep=cfg.separator, header=cfg.header_row,
                     parse_dates=["utctime"], index_col="utctime",
                     dtype={"Remarks": str}, low_memory=False)
    missing = [c for c in cfg.source_columns if c not in df.columns]
    if missing:
        raise KeyError(f"Missing source columns: {missing}")

    out = df[list(cfg.source_columns)].rename(columns=cfg.source_columns)
    out = out[~out.index.duplicated(keep="first")].sort_index()
    hourly = out.resample("1h").mean()
    log.info("Hourly rows: %s | %s -> %s", f"{len(hourly):,}", hourly.index.min(), hourly.index.max())
    return hourly


df_hourly = load_raw(CFG)

Hourly rows: 55,680 | 2015-08-26 00:00:00 -> 2021-12-31 23:00:00


## Physical cleaning

In [3]:
def clean_and_flag(df: pd.DataFrame, cfg: HelsinkiConfig) -> pd.DataFrame:
    df = df.copy()
    idx = df.index.tz_localize("UTC") if df.index.tz is None else df.index
    df["_elevation"] = get_solarposition(idx, cfg.latitude, cfg.longitude)["elevation"].values
    df["Solar_Elevation"] = df["_elevation"]
    night = df["_elevation"] < 0

    for col in ("Irradiance_POA", "GHI", "Power_Actual_W"):
        df[col] = df[col].clip(lower=0)
        if col != "Power_Actual_W":
            df.loc[df[col] > cfg.max_irradiance, col] = np.nan

    lo, hi = cfg.temp_bounds
    for c in ("Temp_Ambient", "Temp_Module"):
        df[c] = df[c].where(df[c].between(lo, hi))
    lo, hi = cfg.wind_bounds
    df["Wind_Speed"] = df["Wind_Speed"].where(df["Wind_Speed"].between(lo, hi))
    df["Humidity"] = df["Humidity"].where(df["Humidity"].between(0, 100))

    # Night: true zero for power and irradiance
    for col in ("Irradiance_POA", "GHI", "Power_Actual_W"):
        df.loc[night, col] = 0.0

    # Daytime sensor dropouts -> missing
    day_up = df["_elevation"] > cfg.dropout_min_elevation
    for col in ("Irradiance_POA", "GHI"):
        dropout = day_up & (df[col] < 1)
        df.loc[dropout, col] = np.nan
        log.info("Dropouts %-16s %s h", col, f"{int(dropout.sum()):,}")

    # Clearness index (diagnostic only)
    cs = Location(cfg.latitude, cfg.longitude, tz="UTC").get_clearsky(idx, model="ineichen")
    cs_ghi = cs["ghi"].values
    df["Clearness_Index"] = np.where(
        cs_ghi > 20,
        np.clip(df["GHI"].to_numpy(dtype=float) / np.maximum(cs_ghi, 1.0), 0, 1.2),
        0.0)
    return df


df_clean = clean_and_flag(df_hourly, CFG)

Dropouts Irradiance_POA   0 h


Dropouts GHI              1 h


## Build the six years and impute gaps

In [4]:
def impute_with_tracking(df: pd.DataFrame, cols, cfg: HelsinkiConfig) -> pd.DataFrame:
    """Linear interpolation up to 3 h, then the mean of the same hour within ±7 days.
    Every imputed value is flagged in `<col>_Imputed`."""
    df = df.copy()
    hours, doys = df.index.hour, df.index.dayofyear
    for col in cols:
        flag = df[col].isnull()
        df[f"{col}_Imputed"] = flag
        if not flag.any():
            continue
        df[col] = df[col].interpolate(method="linear", limit=cfg.interpolate_limit_hours)
        for idx in df.index[df[col].isnull()]:
            near = (hours == idx.hour) & (np.abs(doys - idx.dayofyear) <= cfg.climatology_window_days)
            df.loc[idx, col] = df.loc[near, col].mean()
    return df


def build_years(df: pd.DataFrame, cfg: HelsinkiConfig) -> pd.DataFrame:
    all_cols = list(cfg.model_features) + ["Power_Actual_W"] + list(cfg.diagnostic_only)
    frames = []
    for year in cfg.years:
        full = pd.date_range(f"{year}-01-01 00:00", f"{year}-12-31 23:00",
                             freq="h", tz=df.index.tz)
        y = df[df.index.year == year].reindex(full)
        y = impute_with_tracking(y, all_cols, cfg)      # within the same year only
        y["Site_ID"] = cfg.site_name
        y["Year"] = year
        frames.append(y)

    out = pd.concat(frames).sort_index()
    out["Power_PU"] = (out["Power_Actual_W"] / cfg.nominal_capacity_w).clip(0, 1)
    hour, month = out.index.hour, out.index.month
    out["HOUR_SIN"] = np.sin(2 * np.pi * hour / 24.0)
    out["HOUR_COS"] = np.cos(2 * np.pi * hour / 24.0)
    out["MONTH_SIN"] = np.sin(2 * np.pi * month / 12.0)
    out["MONTH_COS"] = np.cos(2 * np.pi * month / 12.0)

    out = out.drop(columns=["_elevation"], errors="ignore")
    out.index.name = "utctime"
    out = out.reset_index()
    schema = (["utctime", "Site_ID", "Year", "Power_Actual_W", "Power_PU"]
              + list(cfg.model_features)
              + ["HOUR_SIN", "HOUR_COS", "MONTH_SIN", "MONTH_COS"]
              + list(cfg.diagnostic_only)
              + [c for c in out.columns if c.endswith("_Imputed")])
    return out[[c for c in schema if c in out.columns]]


df_final = build_years(df_clean, CFG)

## Checks and save

In [5]:
core = ["Power_Actual_W", "Power_PU"] + list(CFG.model_features)
print("Missing values in core columns:", int(df_final[core].isnull().sum().sum()))
print(df_final["Year"].value_counts().sort_index().to_string())
for c in [c for c in df_final.columns if c.endswith("_Imputed")]:
    print(f"{c.replace('_Imputed', ''):20s} imputed {df_final[c].mean() * 100:6.2f}%")

os.makedirs(os.path.dirname(CFG.output_path), exist_ok=True)
df_final.to_csv(CFG.output_path, index=False)
print("Saved:", CFG.output_path, df_final.shape)

Missing values in core columns: 0
Year
2016    8784
2017    8760
2018    8760
2019    8760
2020    8784
2021    8760
Irradiance_POA       imputed   0.17%
GHI                  imputed   0.02%
Temp_Ambient         imputed   0.35%
Wind_Speed           imputed   0.14%
Humidity             imputed   0.05%
Power_Actual_W       imputed   2.21%
Temp_Module          imputed   3.77%
Solar_Elevation      imputed   0.00%
Clearness_Index      imputed   0.02%
Snow_Indicator       imputed   0.00%


Saved: data/Helsinki_MultiYear.csv (52608, 28)
